# Emergency no-Drive continuation — photo → pencil

Use this only while Google Drive mounting says **credential propagation was unsuccessful**.

This notebook does **not** mount Drive and does **not** restart training. It reconstructs the same deterministic FS2K + 20K COCO data order, accepts your saved `photo_to_pencil/last_training_state.pt`, and resumes from its recorded step. Your completed sketch→photo result remains in the old Drive and is not retrained here.

Important: Colab local storage is temporary. After the training cell completes—or after you stop it safely—run the **Download recovery package** cell before closing the runtime.


## 1. Install and confirm the T4 GPU

In [ ]:
# Install only the packages needed by this recovery notebook.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'diffusers==0.35.1', 'transformers==4.55.4', 'peft==0.17.1',
    'accelerate==1.10.1', 'huggingface-hub==0.34.4', 'safetensors==0.6.2',
    'opencv-python-headless==4.11.0.86', 'Pillow>=10', 'matplotlib',
    'requests', 'tqdm', 'scikit-image==0.25.2', 'lpips==0.1.4', 'gdown==5.2.0'])
print('Installation complete. Continue to the next cell.')


In [ ]:
from pathlib import Path
import os, sys, json, csv, math, random, hashlib, time, gc, shutil, zipfile, io
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import requests
import cv2
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image, ImageOps, ImageEnhance, ImageFilter
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from diffusers import StableDiffusionInstructPix2PixPipeline, DDPMScheduler
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict

assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > T4 GPU before training.'
DEVICE = 'cuda'
DTYPE = torch.float16
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False

import torchvision, subprocess, platform
from collections import Counter, defaultdict
from contextlib import contextmanager
from torch import nn
import lpips

print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB')
print('Local disk free:', round(shutil.disk_usage('/content').free / 2**30, 1), 'GiB')


## 2. Local setup (no Drive mount)

In [ ]:
# NO GOOGLE DRIVE IS USED IN THIS NOTEBOOK.
# Everything is written to temporary Colab storage, so download the recovery ZIP
# before disconnecting the runtime.
MODE = 'emergency_no_drive_resume'
DATA_SOURCE = 'fs2k_faces_plus_coco20k'
DOWNLOAD_FS2K = True
OUTPUT_MODES = {'photo_to_pencil': 'residual', 'sketch_to_photo': 'direct'}

CFG = {
    'resolution':512,'seed':SEED,'cnn_width':64,
    'generator_lr_by_direction':{'photo_to_pencil':5e-5,'sketch_to_photo':5e-5},
    'discriminator_lr_by_direction':{'photo_to_pencil':1e-4,'sketch_to_photo':1e-4},
    'cnn_weight_decay':1e-4,'cnn_validate_every':512,'cnn_save_every':512,
    'metric_images_per_domain':16,
    'photo_aug_probability':0.30,'sketch_aug_probability':0.45,
    'pencil_strength':0.95,'pencil_background_fade':0.25,'pencil_texture':0.30,
    'l1_weight_by_direction':{'photo_to_pencil':6.0,'sketch_to_photo':6.0},
    'perceptual_weight_by_direction':{'photo_to_pencil':2.0,'sketch_to_photo':3.0},
    'gradient_weight_by_direction':{'photo_to_pencil':5.0,'sketch_to_photo':3.0},
    'laplacian_weight_by_direction':{'photo_to_pencil':2.0,'sketch_to_photo':1.5},
    'adversarial_weight_by_direction':{'photo_to_pencil':0.10,'sketch_to_photo':0.15},
    'pencil_gray_weight':3.0,'structure_weight':0.75,
    'chroma_weight':8.0,'color_stats_weight':3.0,'inference_steps':30,
}

SAVE_ROOT = Path('/content/Sketch2Photo_emergency')
CACHE = Path('/content/sketch2photo_data')
RUN_ROOT = SAVE_ROOT / 'working'
STUDY_ROOT = RUN_ROOT
FS2K_ROOT = '/content/FS2K'
for folder in (SAVE_ROOT, CACHE, RUN_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

IMPLEMENTATION_SHA = '911776b154fb5c248c38ca72141f20fa807afdb4d3fb958f369a22c3b7061c76'
atomic_environment = {
    'python': sys.version, 'torch': torch.__version__, 'torchvision': torchvision.__version__,
    'gpu': torch.cuda.get_device_name(0), 'platform': platform.platform(),
    'vram_gib': torch.cuda.get_device_properties(0).total_memory / 2**30,
}
print('Emergency local root:', SAVE_ROOT)
print('Drive mount is intentionally disabled.')


## 3. Upload the interrupted photo checkpoints

From the old Drive website, download the two files named in the cell comment **one at a time** so Drive does not wrap them in a ZIP. Then select both `.pt` files together when the Colab upload box opens.

In [ ]:
# Download these TWO files from the old Drive in your normal browser first:
# Sketch2Photo_training/general_20k_bidirectional_v1/general_rescue/
# unet_multiscale_patchgan/photo_to_pencil/last_training_state.pt
# unet_multiscale_patchgan/photo_to_pencil/best.pt
# Then select both files together in the upload box opened by this cell.
from google.colab import files

PHOTO_FOLDER = SAVE_ROOT / 'general_20k_bidirectional_v1' / 'general_rescue' / 'unet_multiscale_patchgan' / 'photo_to_pencil'
PHOTO_FOLDER.mkdir(parents=True, exist_ok=True)

print('Select photo_to_pencil/last_training_state.pt AND photo_to_pencil/best.pt')
uploaded = files.upload()
assert {'last_training_state.pt', 'best.pt'} <= set(uploaded), (
    'Select both exact files: last_training_state.pt and best.pt'
)
resume_path = PHOTO_FOLDER / 'last_training_state.pt'
shutil.move('last_training_state.pt', resume_path)
shutil.move('best.pt', PHOTO_FOLDER / 'best.pt')
del uploaded

resume_preview = torch.load(resume_path, map_location='cpu', weights_only=False)
assert {'generator', 'discriminator_full', 'discriminator_half', 'opt_g', 'opt_d', 'progress'} <= resume_preview.keys(), (
    'This is not the general photo_to_pencil last_training_state.pt file.'
)
best_preview = torch.load(PHOTO_FOLDER/'best.pt', map_location='cpu', weights_only=True)
assert best_preview.get('direction') == 'photo_to_pencil', 'best.pt is not the photo_to_pencil checkpoint.'
print('Resume checkpoint accepted. Saved step:', resume_preview['progress']['step'])
print('Existing best checkpoint preserved. Best step:', best_preview.get('step'))
assert 0 < resume_preview['progress']['step'] < 21694, 'Checkpoint is already complete or is the wrong file.'
del resume_preview, best_preview


## 4. Restore model and data code

In [ ]:
MODULE_SOURCES = {'src/__init__.py': '"""Local Sketch2Photo application and checkpoint-compatible model definitions."""\n',
 'src/approaches/__init__.py': '"""Architectures for the new, paired 512px comparison (not legacy '
                               'weights)."""\n'
                               '\n'
                               'from .unet import PencilUNet\n'
                               'from .gan import ResnetGenerator, PatchDiscriminator\n'
                               '\n'
                               '\n'
                               'def make_generator(approach, width=64, output_mode="residual"):\n'
                               "    if approach == 'unet':\n"
                               '        return PencilUNet(width, output_mode=output_mode)\n'
                               "    if approach == 'gan':\n"
                               '        return ResnetGenerator(width)\n'
                               '    raise ValueError(approach)\n',
 'src/approaches/diffusion.py': '"""Approach C: LoRA adaptation and InstructPix2Pix training objective.\n'
                                '\n'
                                'The notebook owns shared data, validation and resumable training '
                                'orchestration.\n'
                                'Target latents are VAE-scaled; source-image conditioning latents MUST '
                                'remain unscaled.\n'
                                '"""\n'
                                'import torch\n'
                                '\n'
                                '\n'
                                'def configure_adapter(pipe, rank):\n'
                                '    from peft import LoraConfig, get_peft_model\n'
                                '    assert pipe.unet.config.in_channels == 8\n'
                                '    pipe.vae.requires_grad_(False).eval().to(dtype=torch.float32)\n'
                                '    pipe.text_encoder.requires_grad_(False).eval()\n'
                                '    pipe.unet.requires_grad_(False)\n'
                                '    config = LoraConfig(r=rank, lora_alpha=rank, '
                                "init_lora_weights='gaussian',\n"
                                "                       target_modules=['to_q', 'to_k', 'to_v', 'to_out.0', "
                                "'conv_in'])\n"
                                '    pipe.unet = get_peft_model(pipe.unet, config)\n'
                                '    pipe.unet.enable_gradient_checkpointing()\n'
                                '    for param in pipe.unet.parameters():\n'
                                '        if param.requires_grad:\n'
                                '            param.data = param.data.float()\n'
                                '    return pipe\n'
                                '\n'
                                '\n'
                                'def noise_prediction_loss(pipe, scheduler, embeddings, batch, *, device, '
                                'dtype,\n'
                                '                          conditioning_dropout=.05, snr_gamma=5., '
                                'training=True,\n'
                                '                          fixed_seed=None, timestep=None):\n'
                                '    generator = None if fixed_seed is None else '
                                'torch.Generator(device=device).manual_seed(fixed_seed)\n'
                                "    source = batch['source'].to(device, dtype=torch.float32)\n"
                                "    target = batch['target'].to(device, dtype=torch.float32)\n"
                                '    with torch.no_grad():\n'
                                '        posterior = pipe.vae.encode(target).latent_dist\n'
                                '        clean = posterior.sample(generator=generator) if training else '
                                'posterior.mode()\n'
                                '        clean = (clean * pipe.vae.config.scaling_factor).to(dtype)\n'
                                '        source_latents = '
                                'pipe.vae.encode(source).latent_dist.mode().to(dtype)\n'
                                '    size = clean.shape[0]\n'
                                '    noise = torch.randn(clean.shape, generator=generator, device=device, '
                                'dtype=dtype)\n'
                                '    timesteps = (torch.randint(scheduler.config.num_train_timesteps, '
                                '(size,), device=device, generator=generator)\n'
                                '                 if timestep is None else torch.full((size,), timestep, '
                                'device=device, dtype=torch.long))\n'
                                '    noisy = scheduler.add_noise(clean, noise, timesteps)\n'
                                '    text = embeddings[:1].expand(size, -1, -1)\n'
                                '    if training:\n'
                                '        p = conditioning_dropout\n'
                                '        draw = torch.rand(size, device=device)\n'
                                '        text = torch.where((draw < 2*p).view(size, 1, 1), '
                                'embeddings[1:].expand_as(text), text)\n'
                                '        source_latents = source_latents * (~((draw >= p) & (draw < '
                                '3*p))).view(size, 1, 1, 1)\n'
                                "    with torch.autocast(device_type=str(device).split(':')[0], dtype=dtype, "
                                'enabled=dtype != torch.float32):\n'
                                '        prediction = pipe.unet(torch.cat([noisy, source_latents], dim=1), '
                                'timesteps,\n'
                                '                               encoder_hidden_states=text).sample\n'
                                '    prediction_type = scheduler.config.prediction_type\n'
                                "    if prediction_type == 'epsilon':\n"
                                '        objective = noise\n'
                                "    elif prediction_type == 'v_prediction':\n"
                                '        objective = scheduler.get_velocity(clean, noise, timesteps)\n'
                                '    else:\n'
                                '        raise ValueError(prediction_type)\n'
                                '    loss = (prediction.float()-objective.float()).square().mean(dim=(1, 2, '
                                '3))\n'
                                '    alpha = scheduler.alphas_cumprod.to(device)[timesteps]\n'
                                '    snr = alpha/(1-alpha).clamp_min(1e-8)\n'
                                '    weights = snr.clamp(max=snr_gamma)/(snr+(prediction_type == '
                                "'v_prediction')).clamp_min(1e-8)\n"
                                '    return (loss*weights).mean()\n',
 'src/approaches/gan.py': '"""Approach B: residual generator plus conditional PatchGAN; no U-Net skips.\n'
                          '\n'
                          'Uses adversarial, paired reconstruction and perceptual losses in the notebook.\n'
                          'Inspired by the pix2pix/CycleGAN project; this is a separate PyTorch '
                          'implementation.\n'
                          '"""\n'
                          'from torch import nn\n'
                          '\n'
                          '\n'
                          'class Residual(nn.Module):\n'
                          '    def __init__(self, channels):\n'
                          '        super().__init__()\n'
                          '        self.layers = nn.Sequential(nn.ReflectionPad2d(1), nn.Conv2d(channels, '
                          'channels, 3),\n'
                          '            nn.InstanceNorm2d(channels, affine=True), nn.ReLU(), '
                          'nn.ReflectionPad2d(1),\n'
                          '            nn.Conv2d(channels, channels, 3), nn.InstanceNorm2d(channels, '
                          'affine=True))\n'
                          '\n'
                          '    def forward(self, x):\n'
                          '        return x + self.layers(x)\n'
                          '\n'
                          '\n'
                          'class ResnetGenerator(nn.Module):\n'
                          '    def __init__(self, width=64):\n'
                          '        super().__init__()\n'
                          '        layers = [nn.ReflectionPad2d(3), nn.Conv2d(3, width, 7),\n'
                          '                  nn.InstanceNorm2d(width, affine=True), nn.ReLU()]\n'
                          '        for a, b in ((width, width*2), (width*2, width*4)):\n'
                          '            layers += [nn.Conv2d(a, b, 3, stride=2, padding=1),\n'
                          '                       nn.InstanceNorm2d(b, affine=True), nn.ReLU()]\n'
                          '        layers += [Residual(width*4) for _ in range(9)]\n'
                          '        for a, b in ((width*4, width*2), (width*2, width)):\n'
                          "            layers += [nn.Upsample(scale_factor=2, mode='bilinear', "
                          'align_corners=False),\n'
                          '                       nn.Conv2d(a, b, 3, padding=1), nn.InstanceNorm2d(b, '
                          'affine=True), nn.ReLU()]\n'
                          '        layers += [nn.ReflectionPad2d(3), nn.Conv2d(width, 3, 7), nn.Tanh()]\n'
                          '        self.layers = nn.Sequential(*layers)\n'
                          '\n'
                          '    def forward(self, x):\n'
                          '        return self.layers(x)\n'
                          '\n'
                          '\n'
                          'class PatchDiscriminator(nn.Module):\n'
                          '    def __init__(self, width=64):\n'
                          '        super().__init__()\n'
                          '        layers = [nn.utils.spectral_norm(nn.Conv2d(6, width, 4, 2, 1)), '
                          'nn.LeakyReLU(.2)]\n'
                          '        for a, b, stride in ((width, width*2, 2), (width*2, width*4, 2), '
                          '(width*4, width*8, 1)):\n'
                          '            layers += [nn.utils.spectral_norm(nn.Conv2d(a, b, 4, stride, 1)), '
                          'nn.LeakyReLU(.2)]\n'
                          '        layers += [nn.utils.spectral_norm(nn.Conv2d(width*8, 1, 4, 1, 1))]\n'
                          '        self.layers = nn.Sequential(*layers)\n'
                          '\n'
                          '    def forward(self, paired_images):\n'
                          '        return self.layers(paired_images)\n',
 'src/approaches/unet.py': '"""Approach A: skip-connected U-Net trained with paired reconstruction '
                           'losses."""\n'
                           'import torch\n'
                           'from torch import nn\n'
                           'from torch.nn import functional as F\n'
                           '\n'
                           '\n'
                           'def block(a, b):\n'
                           '    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.GroupNorm(8, b), '
                           'nn.SiLU(),\n'
                           '                         nn.Conv2d(b, b, 3, padding=1), nn.GroupNorm(8, b), '
                           'nn.SiLU())\n'
                           '\n'
                           '\n'
                           'class PencilUNet(nn.Module):\n'
                           '    def __init__(self, width=64, output_mode="residual"):\n'
                           '        super().__init__()\n'
                           '        if output_mode not in {"residual", "direct"}:\n'
                           '            raise ValueError(f"Unsupported output mode: {output_mode}")\n'
                           '        self.output_mode = output_mode\n'
                           '        self.enc = nn.ModuleList([block(3, width), block(width, width*2),\n'
                           '                                 block(width*2, width*4), block(width*4, '
                           'width*8)])\n'
                           '        self.mid = block(width*8, width*8)\n'
                           '        self.dec = nn.ModuleList([block(width*16, width*4), block(width*8, '
                           'width*2),\n'
                           '                                 block(width*4, width), block(width*2, width)])\n'
                           '        self.out = nn.Conv2d(width, 3, 1)\n'
                           '        if output_mode == "residual":\n'
                           '            # Identity initialization is useful when the input and output have '
                           'similar\n'
                           '            # appearance (photo -> pencil).  It is deliberately not used for '
                           'the\n'
                           '            # direct RGB reconstruction head because that otherwise biases a '
                           'colour\n'
                           '            # model toward the monochrome sketch input.\n'
                           '            nn.init.zeros_(self.out.weight)\n'
                           '            nn.init.zeros_(self.out.bias)\n'
                           '\n'
                           '    def forward(self, x):\n'
                           '        source = x\n'
                           '        skips = []\n'
                           '        for layer in self.enc:\n'
                           '            x = layer(x); skips.append(x); x = F.avg_pool2d(x, 2)\n'
                           '        x = self.mid(x)\n'
                           '        for layer, skip in zip(self.dec, reversed(skips)):\n'
                           "            x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', "
                           'align_corners=False)\n'
                           '            x = layer(torch.cat([x, skip], dim=1))\n'
                           '        prediction = self.out(x)\n'
                           '        if self.output_mode == "residual":\n'
                           '            return (source + prediction).clamp(-1, 1)\n'
                           '        return torch.tanh(prediction)\n',
 'src/diffusion_backend.py': '"""Inference for the separate Colab dual-direction diffusion export.\n'
                             '\n'
                             'Dependencies are loaded only when a model is opened. Legacy CNN inference is '
                             'separate.\n'
                             'The same source is embedded in the standalone Colab notebook and its exported '
                             'bundle.\n'
                             '"""\n'
                             '\n'
                             'import json\n'
                             'from pathlib import Path\n'
                             '\n'
                             "BASE_MODEL = 'timbrooks/instruct-pix2pix'\n"
                             "BASE_REVISION = '31519b5cb02a7fd89b906d88731cd4d6a7bbf88d'\n"
                             'PROMPTS = {\n'
                             "    'photo_to_pencil': (\n"
                             "        'Turn this image into a detailed hand-drawn graphite pencil drawing on "
                             "clean white paper. '\n"
                             "        'Use fine natural pencil strokes, delicate cross-hatching, gentle "
                             "tonal shading and clean highlights; '\n"
                             "        'avoid ink outlines, charcoal blocks, colour and digital-painting "
                             "texture. Preserve identity, anatomy, '\n"
                             "        'pose, proportions and composition.'\n"
                             '    ),\n'
                             "    'sketch_to_photo': (\n"
                             "        'Turn this pencil drawing into a sharp realistic full-colour "
                             "photograph with natural skin tones, '\n"
                             "        'distinct believable object colours, realistic lighting and detailed "
                             "photographic textures; avoid '\n"
                             "        'monochrome, sepia, illustration, plastic skin and blur. Preserve "
                             "identity, anatomy, pose, proportions '\n"
                             "        'and composition.'\n"
                             '    ),\n'
                             '}\n'
                             '\n'
                             '\n'
                             'def rgb_on_white(image):\n'
                             '    from PIL import Image, ImageOps\n'
                             '    image = ImageOps.exif_transpose(image)\n'
                             "    rgba = image.convert('RGBA')\n"
                             "    white = Image.new('RGBA', rgba.size, 'white')\n"
                             "    return Image.alpha_composite(white, rgba).convert('RGB')\n"
                             '\n'
                             '\n'
                             'def fit_canvas(image, size=512):\n'
                             '    """Aspect-preserving resize and padding, also used for training pairs."""\n'
                             '    from PIL import Image\n'
                             '    image = rgb_on_white(image)\n'
                             '    scale = size / max(image.size)\n'
                             '    width, height = [max(1, min(size, round(v * scale))) for v in image.size]\n'
                             '    resized = image.resize((width, height), Image.Resampling.LANCZOS)\n'
                             '    left, top = (size - width) // 2, (size - height) // 2\n'
                             "    canvas = Image.new('RGB', (size, size), 'white')\n"
                             '    canvas.paste(resized, (left, top))\n'
                             '    return canvas, (left, top, left + width, top + height)\n'
                             '\n'
                             '\n'
                             'def open_pipeline(adapter_dir=None, device=None, base_model=BASE_MODEL,\n'
                             '                  revision=BASE_REVISION, offload=False):\n'
                             '    import torch\n'
                             '    from diffusers import StableDiffusionInstructPix2PixPipeline, '
                             'DDIMScheduler\n'
                             '    from peft import PeftModel\n'
                             '    if device is None:\n'
                             "        device = 'cuda' if torch.cuda.is_available() else (\n"
                             "            'mps' if torch.backends.mps.is_available() else 'cpu')\n"
                             "    dtype = torch.float16 if device == 'cuda' else torch.float32\n"
                             '    pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(\n'
                             '        base_model, revision=revision, torch_dtype=dtype, '
                             'use_safetensors=True,\n'
                             "        variant='fp16' if dtype == torch.float16 else None)\n"
                             '    if adapter_dir is not None:\n'
                             '        adapter_dir = Path(adapter_dir)\n'
                             "        if not (adapter_dir / 'adapter_model.safetensors').is_file():\n"
                             "            raise FileNotFoundError(f'No trained adapter at {adapter_dir}')\n"
                             '        pipe.unet = PeftModel.from_pretrained(pipe.unet, str(adapter_dir), '
                             'is_trainable=False)\n'
                             '        pipe.unet.to(dtype=dtype)\n'
                             '    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)\n'
                             '    pipe.enable_vae_slicing()\n'
                             '    pipe.set_progress_bar_config(disable=True)\n'
                             "    if offload and device == 'cuda':\n"
                             '        pipe.enable_model_cpu_offload()\n'
                             '    else:\n'
                             '        pipe.to(device)\n'
                             '    return pipe\n'
                             '\n'
                             '\n'
                             'def generate(pipe, image, direction, seed=42, steps=30, text_guidance=6.5,\n'
                             "             image_guidance=2.2, size=512, description=''):\n"
                             '    import torch\n'
                             '    if direction not in PROMPTS:\n'
                             "        raise ValueError(f'Unknown direction: {direction}')\n"
                             "    if direction == 'sketch_to_photo':\n"
                             "        image = rgb_on_white(image).convert('L').convert('RGB')\n"
                             '    canvas, box = fit_canvas(image, size)\n'
                             '    # CPU generator works across CPU, CUDA, and MPS and is repeatable on each '
                             'backend.\n'
                             "    generator = torch.Generator(device='cpu').manual_seed(int(seed))\n"
                             '    prompt = PROMPTS[direction]\n'
                             '    if description.strip():\n'
                             "        prompt += ' ' + description.strip()\n"
                             '    with torch.inference_mode():\n'
                             '        output = pipe(prompt=prompt, image=canvas, '
                             'num_inference_steps=int(steps),\n'
                             '                      guidance_scale=float(text_guidance),\n'
                             '                      image_guidance_scale=float(image_guidance), '
                             'generator=generator)\n'
                             '    if output.nsfw_content_detected is not None and '
                             'any(output.nsfw_content_detected):\n'
                             "        raise RuntimeError('The base model filtered this output; it is not a "
                             "successful prediction.')\n"
                             '    result = output.images[0].crop(box)\n'
                             "    if direction == 'photo_to_pencil':\n"
                             "        result = result.convert('L').convert('RGB')\n"
                             '    return result\n'
                             '\n'
                             '\n'
                             'class DualEditor:\n'
                             '    """Load one direction at a time to limit GPU memory."""\n'
                             '    def __init__(self, bundle_dir, device=None):\n'
                             '        self.root = Path(bundle_dir)\n'
                             "        self.manifest = json.loads((self.root / 'manifest.json').read_text())\n"
                             "        if self.manifest.get('format') != 'sketch2photo-ip2p-lora-v1':\n"
                             "            raise ValueError('Expected a dual-direction notebook export, not a "
                             "legacy .pt checkpoint.')\n"
                             '        self.device = device\n'
                             '        self.pipe = None\n'
                             '        self.direction = None\n'
                             '\n'
                             "    def __call__(self, image, direction, seed=42, description=''):\n"
                             '        import gc\n'
                             '        import torch\n'
                             '        if image is None:\n'
                             "            raise ValueError('Upload an image first.')\n"
                             '        if direction not in PROMPTS:\n'
                             "            raise ValueError('Choose a supported direction.')\n"
                             '        if self.direction != direction:\n'
                             '            self.pipe = None\n'
                             '            gc.collect()\n'
                             '            if torch.cuda.is_available():\n'
                             '                torch.cuda.empty_cache()\n'
                             "            if hasattr(torch, 'mps') and torch.backends.mps.is_available():\n"
                             '                torch.mps.empty_cache()\n'
                             '            self.pipe = open_pipeline(self.root / direction, self.device,\n'
                             "                                      self.manifest['base_model'], "
                             "self.manifest['base_revision'],\n"
                             '                                      offload=True)\n'
                             '            self.direction = direction\n'
                             "        settings = self.manifest['inference']\n"
                             '        return generate(self.pipe, image, direction, seed=seed, '
                             'description=description, **settings)\n'
                             '\n'
                             '\n'
                             'def launch_demo(bundle_dir, port=8094):\n'
                             '    import gradio as gr\n'
                             '    editor = DualEditor(bundle_dir)\n'
                             "    with gr.Blocks(title='Sketch2Photo — trained diffusion adapters') as "
                             'demo:\n'
                             "        gr.Markdown('# Sketch2Photo\\nGenerate a pencil drawing or a colour "
                             "photograph.')\n"
                             "        direction = gr.Radio(list(PROMPTS), value='photo_to_pencil', "
                             "label='Direction')\n"
                             '        with gr.Row():\n'
                             "            source = gr.Image(type='pil', label='Your image')\n"
                             "            result = gr.Image(type='pil', label='Generated result', "
                             "interactive=False, format='png')\n"
                             "        description = gr.Textbox(label='Optional description or desired "
                             "colours')\n"
                             "        seed = gr.Number(value=42, precision=0, label='Seed')\n"
                             "        gr.Button('Generate', variant='primary').click(\n"
                             '            editor, [source, direction, seed, description], result, '
                             'concurrency_limit=1)\n'
                             "        gr.Markdown('Colours are plausible estimates. Fine facial details can "
                             "change. '\n"
                             "                    'Results are generated from the uploaded image using your "
                             "selected trained adapter.')\n"
                             '    demo.queue(default_concurrency_limit=1).launch(\n'
                             "        server_name='127.0.0.1', server_port=int(port), show_error=True)\n"
                             '\n'
                             '\n'
                             "if __name__ == '__main__':\n"
                             '    import argparse\n'
                             '    parser = argparse.ArgumentParser()\n'
                             "    parser.add_argument('--bundle', required=True)\n"
                             "    parser.add_argument('--port', type=int, default=8094)\n"
                             '    args = parser.parse_args()\n'
                             '    launch_demo(args.bundle, args.port)\n',
 'src/study_backend.py': '"""Shared inference for the three-approach Colab study export."""\n'
                         'import gc\n'
                         'import json\n'
                         'from pathlib import Path\n'
                         'from .diffusion_backend import PROMPTS, fit_canvas, open_pipeline, generate\n'
                         '\n'
                         '\n'
                         'class StudyEditor:\n'
                         '    def __init__(self, bundle_dir, device=None):\n'
                         '        import torch\n'
                         '        self.root = Path(bundle_dir)\n'
                         "        self.manifest = json.loads((self.root / 'manifest.json').read_text())\n"
                         "        if self.manifest.get('format') != 'sketch2photo-study-v2':\n"
                         "            raise ValueError('Expected the master notebook study export.')\n"
                         "        self.device = device or ('cuda' if torch.cuda.is_available() else (\n"
                         "            'mps' if hasattr(torch.backends, 'mps') and "
                         "torch.backends.mps.is_available() else 'cpu'))\n"
                         '        self.model, self.key = None, None\n'
                         '\n'
                         "    def __call__(self, image, direction, approach='validation_best', seed=42, "
                         "description=''):\n"
                         '        import numpy as np\n'
                         '        import torch\n'
                         '        from PIL import Image\n'
                         '        from .approaches import make_generator\n'
                         '        if image is None:\n'
                         "            raise ValueError('Upload a photo or pencil sketch first.')\n"
                         '        if direction not in PROMPTS:\n'
                         "            raise ValueError('Unknown direction')\n"
                         "        if approach == 'validation_best':\n"
                         "            approach = self.manifest['validation_best'][direction]\n"
                         '        key = (direction, approach)\n'
                         "        info = self.manifest['models'][direction][approach]\n"
                         '        if key != self.key:\n'
                         '            self.model = None; self.key = None; gc.collect()\n'
                         '            if torch.cuda.is_available():\n'
                         '                torch.cuda.empty_cache()\n'
                         "            if hasattr(torch.backends, 'mps') and "
                         'torch.backends.mps.is_available():\n'
                         '                torch.mps.empty_cache()\n'
                         "            if approach == 'diffusion':\n"
                         "                self.model = open_pipeline(self.root / info['path'], self.device,\n"
                         "                    self.manifest['base_model'], self.manifest['base_revision'], "
                         'offload=True)\n'
                         '            else:\n'
                         "                state = torch.load(self.root / info['path'], map_location='cpu', "
                         'weights_only=True)\n'
                         "                assert state['approach'] == approach and state['direction'] == "
                         'direction\n'
                         "                self.model = make_generator(approach, state['width'])\n"
                         "                self.model.load_state_dict(state['generator'], strict=True)\n"
                         '                self.model.to(self.device).eval()\n'
                         '            self.key = key\n'
                         "        settings = self.manifest['inference']\n"
                         "        if approach == 'diffusion':\n"
                         '            return generate(self.model, image, direction, seed=seed, '
                         'description=description, **settings)\n'
                         "        canvas, box = fit_canvas(image, settings['size'])\n"
                         "        if direction == 'sketch_to_photo':\n"
                         "            canvas = canvas.convert('L').convert('RGB')\n"
                         '        x = torch.from_numpy(np.asarray(canvas, '
                         'dtype=np.float32).copy().transpose(2, 0, 1) / 127.5 - 1)\n'
                         '        with torch.inference_mode():\n'
                         '            y = self.model(x[None].to(self.device))[0].float().cpu().clamp(-1, 1)\n'
                         '        image = Image.fromarray(((y.permute(1, 2, '
                         "0).numpy()+1)*127.5).round().astype('uint8')).crop(box)\n"
                         "        return image.convert('L').convert('RGB') if direction == 'photo_to_pencil' "
                         'else image\n'
                         '\n'
                         '\n'
                         'def launch_demo(bundle_dir, port=8088):\n'
                         '    import gradio as gr\n'
                         '    editor = StudyEditor(bundle_dir)\n'
                         "    with gr.Blocks(title='Sketch2Photo — trained model comparison') as demo:\n"
                         "        gr.Markdown('# Sketch2Photo\\nBoth directions use trained models from your "
                         "Colab export. '\n"
                         "                    'Original colours cannot be uniquely recovered from a "
                         "sketch.')\n"
                         "        gr.Markdown('Export mode: **' + editor.manifest['mode'] + '**. Selected "
                         "defaults: `' +\n"
                         "                    json.dumps(editor.manifest['validation_best']) + '`. Best "
                         "means lowest validation LPIPS, not perfect.')\n"
                         "        direction = gr.Radio(list(PROMPTS), value='photo_to_pencil', "
                         "label='Direction')\n"
                         "        approach = gr.Dropdown(['validation_best', 'unet', 'gan', 'diffusion'], "
                         "value='validation_best', label='Trained approach')\n"
                         '        with gr.Row():\n'
                         "            source = gr.Image(type='pil', label='Input photo / white-paper pencil "
                         "sketch')\n"
                         "            result = gr.Image(type='pil', format='png', interactive=False, "
                         "label='Actual generated output — preview and download')\n"
                         "        description = gr.Textbox(label='Optional description / desired colours "
                         "(diffusion only)')\n"
                         "        seed = gr.Number(value=42, precision=0, label='Diffusion seed')\n"
                         "        gr.Button('Generate', variant='primary').click(editor, [source, direction, "
                         'approach, seed, description], result, concurrency_limit=1)\n'
                         '        def send_to_photo(image):\n'
                         "            return image, 'sketch_to_photo'\n"
                         "        gr.Button('Use this output as sketch input').click(send_to_photo, result, "
                         '[source, direction])\n'
                         "        gr.Markdown('No original-photo retrieval, saturation trick, or separate "
                         "dark sketch is used. '\n"
                         "                    'Generation is at the training resolution; upscaling is not "
                         "extra learned detail.')\n"
                         "    demo.queue(default_concurrency_limit=1).launch(server_name='127.0.0.1', "
                         'server_port=int(port), show_error=True)\n'}

MODULE_ROOT = Path('/content/sketch2photo_code_face_v1')
for relative, content in MODULE_SOURCES.items():
    destination = MODULE_ROOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content)
sys.path.insert(0, str(MODULE_ROOT))
from src.approaches import make_generator, PatchDiscriminator
from src.approaches.diffusion import configure_adapter, noise_prediction_loss
from src.diffusion_backend import BASE_MODEL, BASE_REVISION, PROMPTS, fit_canvas, rgb_on_white, open_pipeline, generate
INFERENCE_SOURCE = MODULE_SOURCES['src/diffusion_backend.py']


In [ ]:
def make_clean_line_art(
    photo_img,
    graphite_strength=1.0,
    background_fade=0.25,
    pencil_texture=0.30,
):
    """
    Produce a white-paper graphite drawing using colour-dodge shading,
    fine dark edges and subtle irregular pencil texture.

    This avoids the embossed appearance of the previous renderer.
    """
    source = rgb_on_white(photo_img)

    working = source.copy()
    working.thumbnail((1600, 1600), Image.Resampling.LANCZOS)

    rgb = np.asarray(working.convert('RGB'))
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

    scale = max(working.size) / 924.0
    scale = max(scale, 0.35)

    # Preserve important boundaries while reducing digital colour noise.
    smooth = cv2.bilateralFilter(
        gray,
        d=7,
        sigmaColor=32,
        sigmaSpace=max(3, int(round(7 * scale))),
    )

    # Traditional pencil-sketch colour-dodge foundation.
    inverted = 255 - smooth
    blurred_inverted = cv2.GaussianBlur(
        inverted,
        (0, 0),
        sigmaX=max(2.0, 6.0 * scale),
        sigmaY=max(2.0, 6.0 * scale),
    )

    denominator = np.maximum(
        255 - blurred_inverted,
        1,
    ).astype(np.uint8)

    dodge = cv2.divide(
        smooth,
        denominator,
        scale=256,
    ).astype(np.float32)

    dodge = np.clip(dodge, 0, 255)

    # Fine and medium lines: hair, eyes, clothing folds and object edges.
    fine_blur = cv2.GaussianBlur(
        smooth,
        (0, 0),
        sigmaX=max(0.45, 0.65 * scale),
    ).astype(np.float32)

    medium_blur = cv2.GaussianBlur(
        smooth,
        (0, 0),
        sigmaX=max(0.9, 1.65 * scale),
    ).astype(np.float32)

    dark_edges = np.maximum(
        medium_blur - fine_blur,
        0.0,
    ) / 255.0

    # Gentle graphite shading based on original luminance.
    shadow = np.power(
        (255.0 - smooth.astype(np.float32)) / 255.0,
        1.45,
    )

    dodge_density = np.clip(
        (255.0 - dodge) / 255.0,
        0.0,
        1.0,
    )

    density = (
        0.74 * dodge_density
        + 0.22 * shadow
        + 2.10 * dark_edges
    )

    # Fade only smooth, low-detail regions. Do not erase subject edges.
    activity = cv2.GaussianBlur(
        dark_edges,
        (0, 0),
        sigmaX=max(1.5, 4.0 * scale),
    )

    activity_reference = float(
        np.percentile(activity, 88)
    )

    if activity_reference > 1e-6:
        focus = np.clip(
            activity / activity_reference,
            0.0,
            1.0,
        )
    else:
        focus = np.ones_like(activity)

    fade = float(
        np.clip(background_fade, 0.0, 1.0)
    )

    # Keep most tonal information; background fade is intentionally mild.
    density *= (
        1.0
        - 0.30 * fade
        + 0.30 * fade * focus
    )

    strength = float(
        np.clip(graphite_strength, 0.65, 1.60)
    )

    density *= strength

    # Add subtle irregular graphite and directional marks.
    texture = float(
        np.clip(pencil_texture, 0.0, 1.0)
    )

    rng = np.random.default_rng(1947)

    grain = rng.normal(
        0.0,
        1.0,
        gray.shape,
    ).astype(np.float32)

    stroke_length = max(
        3,
        int(round(5 * scale)),
    )

    if stroke_length % 2 == 0:
        stroke_length += 1

    stroke_kernel = np.eye(
        stroke_length,
        dtype=np.float32,
    )

    stroke_kernel /= stroke_kernel.sum()

    directional_marks = cv2.filter2D(
        grain,
        -1,
        stroke_kernel,
        borderType=cv2.BORDER_REFLECT,
    )

    directional_marks *= np.sqrt(stroke_length)

    texture_multiplier = np.clip(
        1.0
        + texture * (
            0.055 * grain
            + 0.085 * directional_marks
        ),
        0.75,
        1.30,
    )

    density *= texture_multiplier
    density = np.clip(density, 0.0, 1.6)

    graphite = 255.0 * (
        1.0 - np.exp(-2.15 * density)
    )

    # Slight paper variation, strongest where graphite is present.
    paper_grain = (
        texture
        * 1.3
        * grain
        * np.sqrt(np.clip(graphite / 255.0, 0.0, 1.0))
    )

    paper = 254.0 + 0.25 * texture * grain

    drawing = np.clip(
        paper - graphite + paper_grain,
        0,
        255,
    ).astype(np.uint8)

    # Prevent pure digital black over large regions.
    drawing = np.maximum(drawing, 8).astype(np.uint8)

    result = Image.fromarray(drawing)

    if result.size != source.size:
        result = result.resize(
            source.size,
            Image.Resampling.LANCZOS,
        )

    return result


# New recommended target settings.
CFG['pencil_strength'] = 0.95
CFG['pencil_background_fade'] = 0.25
CFG['pencil_texture'] = 0.30

print('Natural graphite renderer v6 is ready.')

In [ ]:
def atomic_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.tmp')
    temp.write_text(json.dumps(data, indent=2, sort_keys=True))
    os.replace(temp, path)

def file_sha(path):
    with open(path, 'rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

def download_file(url, destination, attempts=3):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file():
        return destination
    temp = destination.with_name(destination.name + '.part')
    for attempt in range(attempts):
        try:
            with requests.get(url, stream=True, timeout=(20, 90)) as response:
                response.raise_for_status()
                with open(temp, 'wb') as f:
                    for chunk in response.iter_content(1024 * 1024):
                        if chunk:
                            f.write(chunk)
                expected = response.headers.get('Content-Length')
                if expected is not None and temp.stat().st_size != int(expected):
                    raise IOError('Incomplete download: ' + url)
            os.replace(temp, destination)
            return destination
        except Exception:
            if attempt == attempts - 1:
                raise
            time.sleep(2 * (attempt + 1))

def public_coco_download(relative, destination):
    # Path-style HTTPS access to the same public COCO S3 bucket avoids the custom
    # hostname's certificate mismatch. TLS verification remains enabled.
    return download_file('https://s3.amazonaws.com/images.cocodataset.org/' + relative, destination)

def build_coco_records():
    annotation_zip = public_coco_download('annotations/annotations_trainval2017.zip', CACHE / 'annotations.zip')
    with zipfile.ZipFile(annotation_zip) as archive:
        metadata = json.loads(archive.read('annotations/captions_train2017.json'))
    # Stable master split: pilot is a strict subset of each full split.
    items = sorted((x for x in metadata['images'] if min(x['width'], x['height']) >= 256 and min(x['width'],x['height'])/max(x['width'],x['height']) >= .125), key=lambda x: x['id'])
    random.Random(SEED).shuffle(items)
    assert len(items) >= 64000
    blocks = {'train': items[:60000], 'val': items[60000:62000], 'test': items[62000:64000]}
    records = []
    for split, values in blocks.items():
        for item in values[:CFG[split + '_count']]:
            records.append({'id': str(item['id']), 'group_id': str(item['id']), 'split': split,
                            'photo': str(CACHE / 'photos' / item['file_name']), 'sketch': '',
                            'relative_url': 'train2017/' + item['file_name']})
    def fetch(record):
        path = public_coco_download(record['relative_url'], record['photo'])
        with Image.open(path) as image:
            image.verify()
        record['photo_sha256'] = file_sha(path)
        return record
    with ThreadPoolExecutor(max_workers=8) as pool:
        records = list(tqdm(pool.map(fetch, records), total=len(records), desc='Original COCO photos'))
    # Exact duplicate source bytes must not leak across splits. Drop all members of
    # a duplicate group across splits and report actual remaining counts.
    hash_splits = {}
    for record in records:
        hash_splits.setdefault(record['photo_sha256'], set()).add(record['split'])
    duplicate_hashes = {key for key, values in hash_splits.items() if len(values) > 1}
    if duplicate_hashes:
        print('Removing cross-split exact duplicate groups:', len(duplicate_hashes))
        records = [r for r in records if r['photo_sha256'] not in duplicate_hashes]
    return records

def build_custom_records():
    csv_path = Path(PAIRED_CSV)
    with csv_path.open(newline='') as f:
        rows = list(csv.DictReader(f))
    assert rows and {'photo', 'sketch', 'split', 'group_id'} <= rows[0].keys()
    selected = []
    for split in ['train', 'val', 'test']:
        group = sorted([r for r in rows if r['split'] == split], key=lambda r: (r['group_id'], r['photo']))
        random.Random(SEED).shuffle(group)
        assert group, f'Missing {split} rows in custom CSV'
        selected.extend(group[:CFG[split + '_count']])
    result = []
    for idx, row in enumerate(tqdm(selected, desc='Verify paired files')):
        assert row['group_id'].strip(), 'Every pair needs a group_id for split checks'
        record = {'id': str(idx), 'group_id': row['group_id'], 'split': row['split']}
        sizes = []
        for key in ('photo', 'sketch'):
            path = (csv_path.parent / row[key]).resolve()
            assert path.is_file(), path
            with Image.open(path) as image:
                image = ImageOps.exif_transpose(image)
                sizes.append(image.size)
            record[key] = str(path)
            record[key + '_sha256'] = file_sha(path)
        ratios = [w / h for w, h in sizes]
        assert abs(ratios[0] - ratios[1]) < 0.01, f'Pair aspect ratios differ: {record}'
        result.append(record)
    return result


## 5. Download and verify FS2K

In [ ]:
def build_fs2k_records():
    root = Path(FS2K_ROOT)

    # Download and safely extract FS2K when it is not already available.
    if not (root / 'anno_train.json').exists():
        if not DOWNLOAD_FS2K:
            raise FileNotFoundError(
                'Extract official FS2K to FS2K_ROOT or enable DOWNLOAD_FS2K. '
                'See https://github.com/DengPingFan/FS2K'
            )

        import gdown

        archive = CACHE / 'FS2K.zip'

        if not archive.exists():
            gdown.download(
                id='1saIMhQ3dc5_ftkfGmBPbCluRn_zy7QQp',
                output=str(archive),
                quiet=False,
            )

        if not zipfile.is_zipfile(archive):
            raise ValueError(
                'FS2K download is not a ZIP. Download the official archive '
                'manually; no fallback was made.'
            )

        extraction = CACHE / 'fs2k_extracted'
        extraction.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(archive) as z:
            extraction_root = extraction.resolve()

            for entry in z.infolist():
                resolved = (extraction / entry.filename).resolve()

                if (
                    not resolved.is_relative_to(extraction_root)
                    or ((entry.external_attr >> 16) & 0o170000) == 0o120000
                ):
                    raise ValueError('Unsafe ZIP entry')

            z.extractall(extraction)

        candidates = list(extraction.rglob('anno_train.json'))

        assert len(candidates) == 1, (
            'Could not find one unique FS2K dataset folder. '
            'Set FS2K_ROOT manually.'
        )

        root = candidates[0].parent

    train_annotation = root / 'anno_train.json'
    test_annotation = root / 'anno_test.json'

    if not test_annotation.exists():
        raise FileNotFoundError(
            f'FS2K test annotation was not found: {test_annotation}'
        )

    train = json.loads(train_annotation.read_text())
    test = json.loads(test_annotation.read_text())

    # Make deterministic train and validation splits, separated by style.
    style_groups = defaultdict(list)

    for row in train:
        style_groups[str(row['style'])].append(row)

    tagged = []

    for style, rows in sorted(style_groups.items()):
        rows = sorted(rows, key=lambda row: row['image_name'])
        random.Random(SEED).shuffle(rows)

        cut = max(1, round(0.2 * len(rows)))

        tagged.extend((row, 'val') for row in rows[:cut])
        tagged.extend((row, 'train') for row in rows[cut:])

    tagged.extend((row, 'test') for row in test)

    # FS2K can contain uppercase or uncommon image extensions.
    # Build an index from the files that actually exist instead of guessing.
    allowed_extensions = {
        '.jpg',
        '.jpeg',
        '.png',
        '.bmp',
        '.webp',
        '.tif',
        '.tiff',
    }

    def normalize_relative_stem(value):
        value = str(value).replace('\\', '/')

        return (
            Path(value)
            .with_suffix('')
            .as_posix()
            .casefold()
        )

    def build_image_index(folder):
        base = root / folder

        if not base.exists():
            raise FileNotFoundError(
                f'FS2K image folder was not found: {base}'
            )

        index = defaultdict(list)

        for path in base.rglob('*'):
            if path.is_file() and path.suffix.lower() in allowed_extensions:
                relative_path = path.relative_to(base)
                key = normalize_relative_stem(relative_path)
                index[key].append(path)

        assert index, f'No FS2K images were found under {base}'

        return index

    photo_index = build_image_index('photo')
    sketch_index = build_image_index('sketch')

    def resolve_image(index, name):
        key = normalize_relative_stem(name)
        matches = index.get(key, [])

        # Return None for both missing and ambiguous paths.
        # We must never guess because that could create incorrect pairs.
        if len(matches) != 1:
            return None

        return matches[0]

    result = []
    excluded = []

    for row, split in tagged:
        stem = str(row['image_name']).replace('\\', '/')
        stem_without_extension = Path(stem).with_suffix('').as_posix()

        # Example:
        # photo3/image0449 -> sketch3/sketch0449
        parts = stem_without_extension.split('/')

        if not parts:
            excluded.append({
                'image_name': stem,
                'split': split,
                'reason': 'invalid annotation path',
                'photo_matches': 0,
                'sketch_matches': 0,
            })
            continue

        parts[0] = parts[0].replace('photo', 'sketch', 1)
        sketch_stem = '/'.join(parts)

        filename = Path(sketch_stem).name
        parent = Path(sketch_stem).parent

        if filename.startswith('image'):
            filename = 'sketch' + filename[len('image'):]

        sketch_stem = (
            filename
            if str(parent) == '.'
            else (parent / filename).as_posix()
        )

        photo_key = normalize_relative_stem(stem_without_extension)
        sketch_key = normalize_relative_stem(sketch_stem)

        photo = resolve_image(photo_index, stem_without_extension)
        sketch = resolve_image(sketch_index, sketch_stem)

        if photo is None or sketch is None:
            excluded.append({
                'image_name': stem,
                'expected_sketch_name': sketch_stem,
                'split': split,
                'reason': 'missing or ambiguous exact path',
                'photo_matches': len(photo_index.get(photo_key, [])),
                'sketch_matches': len(sketch_index.get(sketch_key, [])),
            })
            continue

        # Confirm that paired images have compatible geometry.
        with Image.open(photo) as photo_image:
            photo_width, photo_height = photo_image.size

        with Image.open(sketch) as sketch_image:
            sketch_width, sketch_height = sketch_image.size

        photo_ratio = photo_width / photo_height
        sketch_ratio = sketch_width / sketch_height

        if abs(photo_ratio - sketch_ratio) >= 0.01:
            excluded.append({
                'image_name': stem,
                'expected_sketch_name': sketch_stem,
                'split': split,
                'reason': 'photo and sketch aspect ratios differ',
                'photo_size': [photo_width, photo_height],
                'sketch_size': [sketch_width, sketch_height],
            })
            continue

        result.append({
            'id': 'fs2k_' + stem_without_extension,
            'group_id': 'fs2k_' + stem_without_extension,
            'split': split,
            'domain': 'fs2k_artist',
            'style': str(row['style']),
            'photo': str(photo),
            'sketch': str(sketch),
            'photo_sha256': file_sha(photo),
            'sketch_sha256': file_sha(sketch),
        })

    # Save a report of annotations that could not be paired safely.
    audit = {
        'dataset_root': str(root),
        'annotated_pairs': len(tagged),
        'usable_pairs': len(result),
        'excluded_count': len(excluded),
        'excluded_pairs': excluded,
    }

    atomic_json(
        STUDY_ROOT / 'fs2k_pair_audit.json',
        audit,
    )

    minimum_required = int(0.98 * len(tagged))

    assert len(result) >= minimum_required, (
        f'Too many FS2K pairs are unavailable: '
        f'{len(result)}/{len(tagged)} usable. '
        f'Inspect {STUDY_ROOT / "fs2k_pair_audit.json"}. '
        'The FS2K archive may be incomplete or incorrectly extracted.'
    )

    print(
        'FS2K usable pairs:',
        len(result),
        '/',
        len(tagged),
        '| excluded:',
        len(excluded),
    )

    if excluded:
        print(
            'Incomplete or ambiguous annotations were safely excluded. '
            'Details: fs2k_pair_audit.json'
        )

    # Use a small deterministic subset only in pilot mode.
    if False:  # face rescue uses every audited FS2K pair
        selected = []

        for split in ('train', 'val', 'test'):
            rows = [
                record
                for record in result
                if record['split'] == split
            ]

            random.Random(SEED).shuffle(rows)
            limit = 256 if split == 'train' else 32
            selected.extend(rows[:limit])

        result = selected

        print('Pilot-mode FS2K records:', len(result))

    return result


# The failed mixed-domain run showed that synthetic scenes hide poor real-face quality.
# Use every audited real FS2K pair and do not download COCO in this correction.
RECORDS = build_fs2k_records()
for record in RECORDS:
    record['domain'] = 'fs2k_artist'

# Detect data leakage across train, validation, and test sets.
seen_groups = {}
seen_files = {}

for record in RECORDS:
    group_id = record['group_id']
    split = record['split']

    assert seen_groups.setdefault(group_id, split) == split, (
        f'Group leakage across splits: {group_id}'
    )

    for key in ('photo_sha256', 'sketch_sha256'):
        if key in record:
            file_hash = record[key]

            assert seen_files.setdefault(file_hash, split) == split, (
                f'Identical file crosses splits: {record["id"]}'
            )

SPLITS = {
    split: [
        record
        for record in RECORDS
        if record['split'] == split
    ]
    for split in ('train', 'val', 'test')
}

assert all(SPLITS.values()), (
    'At least one train, validation, or test split is empty.'
)

DATA_FINGERPRINT = hashlib.sha256(
    json.dumps(RECORDS, sort_keys=True).encode()
).hexdigest()

atomic_json(
    STUDY_ROOT / 'data_manifest.json',
    {
        'source': DATA_SOURCE,
        'sha256': DATA_FINGERPRINT,
        'records': RECORDS,
    },
)

print(
    'Actual split/domain counts:',
    Counter(
        (record['split'], record['domain'])
        for record in RECORDS
    ),
)

print(
    'Real drawing style counts:',
    Counter(
        (
            record['split'],
            record.get('style', 'unknown'),
        )
        for record in RECORDS
        if record['sketch']
    ),
)


def training_order(epoch, direction=None):
    order = list(range(len(SPLITS['train'])))
    random.Random(SEED + epoch).shuffle(order)
    return order

VAL_RECORDS = []

for domain in sorted({
    record['domain']
    for record in SPLITS['val']
}):
    rows = [
        record
        for record in SPLITS['val']
        if record['domain'] == domain
    ]

    random.Random(SEED).shuffle(rows)

    VAL_RECORDS += rows[
        :CFG['metric_images_per_domain']
    ]

atomic_json(
    STUDY_ROOT / 'validation_selection_ids.json',
    [record['id'] for record in VAL_RECORDS],
)

print(
    'Checkpoint selection uses',
    len(VAL_RECORDS),
    'fixed validation images with equal domain weighting.',
)

In [ ]:
def uncached_pair(record):
    with Image.open(record['photo']) as image:
        photo, box = fit_canvas(image, CFG['resolution'])
    if record['sketch']:
        with Image.open(record['sketch']) as image:
            sketch, sketch_box = fit_canvas(image, CFG['resolution'])
        assert sketch_box == box, 'Paired geometry must match after fitting'
        sketch = sketch.convert('L').convert('RGB')
    else:
        sketch = make_clean_line_art(photo, CFG['pencil_strength'],
                                    CFG['pencil_background_fade'], CFG['pencil_texture']).convert('RGB')
    return photo, sketch, box


def get_pair(record):
    key = hashlib.sha256((record['photo_sha256']+record.get('sketch_sha256','')+str(CFG['resolution'])+
        str(CFG['pencil_strength'])+str(CFG['pencil_background_fade'])+str(CFG['pencil_texture'])+'v8_face_specific_all_pairs').encode()).hexdigest()
    folder = CACHE/'fitted_pairs'/key[:2]; folder.mkdir(parents=True, exist_ok=True)
    photo_path, sketch_path, box_path = (folder/(key+s) for s in ('_photo.png','_sketch.png','.json'))
    if not box_path.exists():
        photo, sketch, box = uncached_pair(record)
        assert min(box[2]-box[0],box[3]-box[1])>=64, 'Pair aspect ratio is too extreme for shared LPIPS evaluation'
        for image,path in ((photo,photo_path),(sketch,sketch_path)):
            temp=path.with_name(path.name+f'.{os.getpid()}.tmp')
            image.save(temp,format='PNG'); os.replace(temp,path)
        temp=box_path.with_name(box_path.name+f'.{os.getpid()}.tmp')
        temp.write_text(json.dumps(box)); os.replace(temp,box_path)
    with Image.open(photo_path) as p, Image.open(sketch_path) as s:
        return p.convert('RGB'), s.convert('RGB'), tuple(json.loads(box_path.read_text()))

def tensor_image(image):
    array = np.asarray(image, dtype=np.float32).copy() / 127.5 - 1.0
    return torch.from_numpy(array.transpose(2, 0, 1))

def augment_white_paper_sketch(image, rng):
    """Mild input-only augmentation for real scanner/app pencil variations."""
    sketch = ImageOps.autocontrast(image.convert('L'), cutoff=0.2)
    sketch = ImageEnhance.Contrast(sketch).enhance(rng.uniform(0.85, 1.20))
    sketch = ImageEnhance.Brightness(sketch).enhance(rng.uniform(0.96, 1.04))
    if rng.random() < 0.25:
        sketch = sketch.filter(ImageFilter.MinFilter(3))  # slightly thicker dark strokes
    elif rng.random() < 0.25:
        sketch = sketch.filter(ImageFilter.GaussianBlur(rng.uniform(0.15, 0.55)))
    array = np.asarray(sketch, dtype=np.float32)
    noise = np.random.default_rng(rng.randrange(2**32)).normal(0, rng.uniform(0, 1.5), array.shape)
    return Image.fromarray(np.clip(array + noise, 0, 255).astype(np.uint8)).convert('RGB')


def augment_photo_input(image, rng):
    """Small camera variation without changing geometry or the paired target."""
    image = ImageEnhance.Brightness(image).enhance(rng.uniform(0.96, 1.04))
    image = ImageEnhance.Contrast(image).enhance(rng.uniform(0.94, 1.06))
    return ImageEnhance.Color(image).enhance(rng.uniform(0.90, 1.10))


class PairedDataset(Dataset):
    def __init__(self, records, direction, training=False, epoch=0):
        self.records, self.direction, self.training, self.epoch = records, direction, training, epoch
    def __len__(self):
        return len(self.records)
    def __getitem__(self, index):
        photo, sketch, _ = get_pair(self.records[index])
        # All randomness is derived from seed/epoch/index, so mid-epoch resume is reproducible.
        rng = random.Random(SEED + 1000003 * self.epoch + index)
        if self.training and rng.random() < 0.5:
            photo, sketch = ImageOps.mirror(photo), ImageOps.mirror(sketch)
        if self.direction == 'photo_to_pencil':
            source, target = photo, sketch
            if self.training and rng.random() < CFG['photo_aug_probability']:
                source = augment_photo_input(source, rng)
        else:
            source, target = sketch, photo
            if self.training and rng.random() < CFG['sketch_aug_probability']:
                source = augment_white_paper_sketch(source, rng)
        return {'source': tensor_image(source), 'target': tensor_image(target)}

preview_rows = [
    record
    for domain in sorted({
        record['domain']
        for record in SPLITS['train']
    })
    for record in [
        row
        for row in SPLITS['train']
        if row['domain'] == domain
    ][:6]
]
fig, axes = plt.subplots(len(preview_rows), 2, figsize=(10, 4 * len(preview_rows)))
for i, record in enumerate(preview_rows):
    photo, sketch, box = get_pair(record)
    for ax, im, name in zip(axes[i], [photo.crop(box), sketch.crop(box)], ['Photo target / photo input', 'Pencil target / pencil input']):
        ax.imshow(im); ax.set_title(name); ax.axis('off')
plt.tight_layout()
plt.savefig(RUN_ROOT / 'training_pairs.png', dpi=130)
plt.show()
print('Review these targets before any training. They define the pencil style the models learn.')
if False:  # review is visual but no interactive full-run prompt
    approval = input('Inspect training_pairs.png above. Type TRAIN to start the expensive full run: ').strip()
    if approval != 'TRAIN':
        raise RuntimeError('Training stopped safely before model updates. Fix the data/style, then run again.')
    atomic_json(STUDY_ROOT/'target_review.json', {
        'approved': True, 'typed': approval, 'data_fingerprint': DATA_FINGERPRINT,
        'note': FULL_RUN_REVIEW, 'time_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())})
    print('Target review recorded. Full training is authorized and remains resumable.')


In [ ]:
PERCEPTUAL = lpips.LPIPS(net='alex', version='0.1').to(DEVICE).eval().requires_grad_(False)

def image_from_tensor(tensor, direction):
    array = ((tensor.detach().float().cpu().clamp(-1,1).permute(1,2,0).numpy()+1)*127.5).round().astype('uint8')
    image = Image.fromarray(array)
    return image.convert('L').convert('RGB') if direction == 'photo_to_pencil' else image

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def atomic_torch(path, state):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    torch.save(state, temporary); os.replace(temporary, path)

def luminance(image):
    return 0.299*image[:,0:1] + 0.587*image[:,1:2] + 0.114*image[:,2:3]

def chroma_components(image):
    y = luminance(image); return torch.cat((image[:,2:3]-y, image[:,0:1]-y), dim=1)

def spatial_gradients(image):
    return image[:,:,:,1:]-image[:,:,:,:-1], image[:,:,1:,:]-image[:,:,:-1,:]

def colour_statistics(image):
    return image.mean(dim=(2,3)), image.std(dim=(2,3), unbiased=False)

def laplacian(image):
    kernel = image.new_tensor([[0.,1.,0.],[1.,-4.,1.],[0.,1.,0.]])[None,None]
    return F.conv2d(image, kernel.repeat(image.shape[1],1,1,1), padding=1, groups=image.shape[1])

def paired_direction_loss(pred, target, source, direction):
    pred_dx,pred_dy = spatial_gradients(pred); target_dx,target_dy = spatial_gradients(target)
    loss = (CFG['l1_weight_by_direction'][direction]*F.l1_loss(pred,target)
        + CFG['perceptual_weight_by_direction'][direction]*PERCEPTUAL(pred,target).mean()
        + CFG['gradient_weight_by_direction'][direction]*(F.l1_loss(pred_dx,target_dx)+F.l1_loss(pred_dy,target_dy))
        + CFG['laplacian_weight_by_direction'][direction]*F.l1_loss(laplacian(pred),laplacian(target)))
    if direction == 'photo_to_pencil':
        neutral = F.l1_loss(pred[:,0:1],pred[:,1:2]) + F.l1_loss(pred[:,1:2],pred[:,2:3])
        return loss + CFG['pencil_gray_weight']*neutral
    pred_luma,source_luma = luminance(pred),luminance(source)
    pred_sx,pred_sy = spatial_gradients(pred_luma); source_sx,source_sy = spatial_gradients(source_luma)
    pred_mean,pred_std = colour_statistics(pred); target_mean,target_std = colour_statistics(target)
    return (loss + CFG['structure_weight']*(F.l1_loss(pred_sx,source_sx)+F.l1_loss(pred_sy,source_sy))
        + CFG['chroma_weight']*F.l1_loss(chroma_components(pred),chroma_components(target))
        + CFG['color_stats_weight']*(F.l1_loss(pred_mean,target_mean)+F.l1_loss(pred_std,target_std)))

def load_unet_checkpoint(path):
    state = torch.load(path, map_location='cpu', weights_only=True)
    model = make_generator('unet', state['width'], output_mode=state.get('output_mode','residual')).to(DEVICE)
    model.load_state_dict(state['generator'], strict=True)
    return model.eval(), state

@torch.no_grad()
def predict_unet(model, source, direction):
    return image_from_tensor(model(tensor_image(source)[None].to(DEVICE))[0], direction)


## 6. Rebuild the same persistent 20K COCO subset

This downloads 20,512 images again because Drive is unavailable. It normally takes about 25–40 minutes and uses temporary Colab disk.

In [ ]:
GENERAL_EXPERIMENT = 'general_20k_bidirectional_v1'
GENERAL_ROOT = SAVE_ROOT / GENERAL_EXPERIMENT / 'general_rescue'
GENERAL_DATA_ROOT = Path('/content/datasets/coco2017_general_20k')
GENERAL_PHOTO_ROOT = GENERAL_DATA_ROOT / 'photos'
GENERAL_ROOT.mkdir(parents=True, exist_ok=True)
GENERAL_PHOTO_ROOT.mkdir(parents=True, exist_ok=True)

GENERAL_CFG = {
    'train_count': 20000,
    'val_count': 256,
    'test_count': 256,
    'fs2k_replays': 2,
    'progressive_512_start': 12000,
    'validate_every': 2000,
    'save_every': 500,
    'generator_lr_by_direction': {'sketch_to_photo': 2e-5, 'photo_to_pencil': 1e-5},
    'discriminator_lr': 1e-4,
    'weight_decay': 1e-4,
    'adversarial_max_by_direction': {'sketch_to_photo': 1.0, 'photo_to_pencil': .5},
    'adversarial_warmup': 3000,
    'validation_coco': 16,
    'validation_fs2k': 16,
    'direction_order': ['photo_to_pencil'],
}
print('This cell downloads the same fixed 20,512 COCO images used by the interrupted run.')
print('It does not retrain sketch_to_photo and does not start photo training from zero.')


In [ ]:
def build_persistent_general_coco_records():
    annotation_zip = public_coco_download(
        'annotations/annotations_trainval2017.zip',
        GENERAL_DATA_ROOT / 'annotations_trainval2017.zip',
    )
    with zipfile.ZipFile(annotation_zip) as archive:
        metadata = json.loads(archive.read('annotations/captions_train2017.json'))

    items = sorted(
        (
            item for item in metadata['images']
            if min(item['width'], item['height']) >= 256
            and min(item['width'], item['height']) / max(item['width'], item['height']) >= .125
        ),
        key=lambda item: item['id'],
    )
    random.Random(SEED).shuffle(items)
    assert len(items) >= 64000

    # These boundaries are identical to the original master experiment. Validation
    # and test never overlap the first 60K training pool.
    blocks = {
        'train': items[:GENERAL_CFG['train_count']],
        'val': items[60000:60000 + GENERAL_CFG['val_count']],
        'test': items[62000:62000 + GENERAL_CFG['test_count']],
    }
    records = []
    for split, rows in blocks.items():
        for item in rows:
            records.append({
                'id': 'coco_' + str(item['id']),
                'group_id': 'coco_' + str(item['id']),
                'split': split,
                'domain': 'coco_synthetic',
                'photo': str(GENERAL_PHOTO_ROOT / item['file_name']),
                'sketch': '',
                'relative_url': 'train2017/' + item['file_name'],
            })

    def fetch(record):
        path = public_coco_download(record['relative_url'], record['photo'])
        with Image.open(path) as image:
            image.verify()
        record['photo_sha256'] = file_sha(path)
        return record

    with ThreadPoolExecutor(max_workers=8) as pool:
        records = list(tqdm(
            pool.map(fetch, records),
            total=len(records),
            desc='Persistent COCO 20K + validation/test',
        ))

    # Prove that no exact source file crosses a split.
    seen = {}
    for record in records:
        digest = record['photo_sha256']
        previous = seen.setdefault(digest, record['split'])
        assert previous == record['split'], 'Exact COCO image crosses data splits.'

    manifest = {
        'source': 'COCO 2017 captions image metadata',
        'seed': SEED,
        'split_rule': 'shuffled eligible IDs: train[0:20000], val[60000:60256], test[62000:62256]',
        'counts': {split: len(rows) for split, rows in blocks.items()},
        'records': records,
    }
    atomic_json(GENERAL_DATA_ROOT / 'manifest.json', manifest)
    print('Persistent COCO counts:', Counter(record['split'] for record in records))
    return records


GENERAL_COCO_RECORDS = build_persistent_general_coco_records()
GENERAL_COCO_SPLITS = {
    split: [record for record in GENERAL_COCO_RECORDS if record['split'] == split]
    for split in ('train', 'val', 'test')
}

# Every COCO training image appears once. Real FS2K pairs are replayed twice to
# prevent the broad synthetic stage from erasing the portrait knowledge.
GENERAL_TRAIN_RECORDS = list(GENERAL_COCO_SPLITS['train'])
for _ in range(GENERAL_CFG['fs2k_replays']):
    GENERAL_TRAIN_RECORDS.extend(dict(record) for record in SPLITS['train'])

random.Random(SEED + 700).shuffle(GENERAL_TRAIN_RECORDS)
GENERAL_VAL_RECORDS = (
    GENERAL_COCO_SPLITS['val'][:GENERAL_CFG['validation_coco']]
    + SPLITS['val'][:GENERAL_CFG['validation_fs2k']]
)

GENERAL_DATA_FINGERPRINT = hashlib.sha256(json.dumps({
    'coco': [(record['id'], record['split'], record['photo_sha256']) for record in GENERAL_COCO_RECORDS],
    'fs2k': DATA_FINGERPRINT,
    'replays': GENERAL_CFG['fs2k_replays'],
}, sort_keys=True).encode()).hexdigest()

atomic_json(GENERAL_ROOT / 'training_manifest.json', {
    'unique_coco_train_images': len(GENERAL_COCO_SPLITS['train']),
    'unique_fs2k_train_pairs': len(SPLITS['train']),
    'total_updates': len(GENERAL_TRAIN_RECORDS),
    'validation_coco': len(GENERAL_COCO_SPLITS['val']),
    'test_coco': len(GENERAL_COCO_SPLITS['test']),
    'fingerprint': GENERAL_DATA_FINGERPRINT,
})
print('General training updates:', len(GENERAL_TRAIN_RECORDS))
print('Unique training sources:', len(GENERAL_COCO_SPLITS['train']) + len(SPLITS['train']))


## 7. Resume photo → pencil only

In [ ]:
def get_general_pair(record, resolution=512):
    if record['domain'] == 'fs2k_artist':
        photo, sketch, _ = get_pair(record)
        if photo.size != (resolution, resolution):
            photo = photo.resize((resolution, resolution), Image.Resampling.LANCZOS)
            sketch = sketch.resize((resolution, resolution), Image.Resampling.LANCZOS)
        return photo, sketch
    with Image.open(record['photo']) as image:
        photo, _ = fit_canvas(image, resolution)
    sketch = make_clean_line_art(photo, CFG['pencil_strength'], CFG['pencil_background_fade'], CFG['pencil_texture']).convert('RGB')
    return photo, sketch


class GeneralBidirectionalDataset(Dataset):
    def __init__(self, records, direction, resolution, training=True):
        self.records, self.direction, self.resolution, self.training = records, direction, resolution, training
    def __len__(self): return len(self.records)
    def __getitem__(self, index):
        record = self.records[index]
        photo, sketch = get_general_pair(record, self.resolution)
        rng = random.Random(SEED + 1000003 * index)
        if self.training and rng.random() < .5:
            photo, sketch = ImageOps.mirror(photo), ImageOps.mirror(sketch)
        if self.training and rng.random() < .35:
            sketch = augment_white_paper_sketch(sketch, rng)
        source, target = ((photo, sketch) if self.direction == 'photo_to_pencil' else (sketch, photo))
        return {'source': tensor_image(source), 'target': tensor_image(target), 'domain': record['domain']}


@torch.no_grad()
def general_validation(model, direction):
    model.eval(); rows = []
    for record in GENERAL_VAL_RECORDS:
        photo, sketch = get_general_pair(record, CFG['resolution'])
        source, target = ((photo, sketch) if direction == 'photo_to_pencil' else (sketch, photo))
        source_tensor = tensor_image(source)[None].to(DEVICE)
        target_tensor = tensor_image(target)[None].to(DEVICE)
        prediction = model(source_tensor)
        rows.append({'id':record['id'], 'domain':record['domain'], 'direction':direction,
                     'lpips':float(PERCEPTUAL(prediction,target_tensor).mean()),
                     'mae':float(F.l1_loss(prediction,target_tensor)/2)})
    model.train(); return rows


def train_resumed_photo_to_pencil():
    direction = 'photo_to_pencil'
    seed_everything(SEED)
    folder = PHOTO_FOLDER
    target_updates = len(GENERAL_TRAIN_RECORDS)
    output_mode = OUTPUT_MODES[direction]
    generator = make_generator('unet', CFG['cnn_width'], output_mode=output_mode).to(DEVICE)
    discriminator_full = PatchDiscriminator(CFG['cnn_width']).to(DEVICE)
    discriminator_half = PatchDiscriminator(CFG['cnn_width']).to(DEVICE)
    opt_g = torch.optim.AdamW(generator.parameters(), lr=GENERAL_CFG['generator_lr_by_direction'][direction],
                              weight_decay=GENERAL_CFG['weight_decay'], betas=(.5,.999))
    discriminator_parameters = list(discriminator_full.parameters()) + list(discriminator_half.parameters())
    opt_d = torch.optim.AdamW(discriminator_parameters, lr=GENERAL_CFG['discriminator_lr'],
                              weight_decay=0., betas=(.5,.999))
    sched_g = torch.optim.lr_scheduler.CosineAnnealingLR(opt_g, T_max=target_updates,
        eta_min=GENERAL_CFG['generator_lr_by_direction'][direction]*.1)
    sched_d = torch.optim.lr_scheduler.CosineAnnealingLR(opt_d, T_max=target_updates,
        eta_min=GENERAL_CFG['discriminator_lr']*.1)

    last = folder / 'last_training_state.pt'
    assert last.is_file(), 'Run the checkpoint upload cell first.'
    state = torch.load(last, map_location='cpu', weights_only=False)
    original_signature = state.get('signature', 'uploaded-emergency-resume')
    generator.load_state_dict(state['generator'])
    discriminator_full.load_state_dict(state['discriminator_full'])
    discriminator_half.load_state_dict(state['discriminator_half'])
    opt_g.load_state_dict(state['opt_g']); opt_d.load_state_dict(state['opt_d'])
    sched_g.load_state_dict(state['sched_g']); sched_d.load_state_dict(state['sched_d'])
    progress, history = state['progress'], state['history']
    torch.set_rng_state(state['rng']); torch.cuda.set_rng_state_all(state['cuda_rng'])
    random.setstate(state['python_rng']); np.random.set_state(state['numpy_rng'])
    del state
    print('Resumed photo_to_pencil at step', progress['step'], 'of', target_updates)
    assert progress['step'] <= target_updates, 'Uploaded checkpoint has an invalid step.'

    started = time.perf_counter(); elapsed_before = progress.get('seconds', 0.)

    def checkpoint(path, score):
        atomic_torch(path, {'generator':{k:v.detach().cpu() for k,v in generator.state_dict().items()},
            'approach':'unet', 'training':'general_20k_multiscale_patchgan', 'direction':direction,
            'output_mode':output_mode, 'width':CFG['cnn_width'], 'resolution':CFG['resolution'],
            'data_fingerprint':GENERAL_DATA_FINGERPRINT,
            'unique_coco_train_images':len(GENERAL_COCO_SPLITS['train']),
            'unique_fs2k_train_pairs':len(SPLITS['train']), 'step':progress['step'],
            'val_lpips':score, 'transferred_from':'uploaded last_training_state.pt'})

    def save_last():
        progress['seconds'] = elapsed_before + time.perf_counter() - started
        atomic_torch(last, {'signature':original_signature, 'generator':generator.state_dict(),
            'discriminator_full':discriminator_full.state_dict(), 'discriminator_half':discriminator_half.state_dict(),
            'opt_g':opt_g.state_dict(), 'opt_d':opt_d.state_dict(), 'sched_g':sched_g.state_dict(),
            'sched_d':sched_d.state_dict(), 'progress':dict(progress), 'history':history,
            'rng':torch.get_rng_state(), 'cuda_rng':torch.cuda.get_rng_state_all(),
            'python_rng':random.getstate(), 'numpy_rng':np.random.get_state()})
        atomic_json(folder/'history.json', history)

    def validate():
        rows = general_validation(generator, direction)
        overall = float(np.mean([row['lpips'] for row in rows]))
        coco = float(np.mean([row['lpips'] for row in rows if row['domain']=='coco_synthetic']))
        fs2k = float(np.mean([row['lpips'] for row in rows if row['domain']=='fs2k_artist']))
        entry = {'step':progress['step'], 'val_lpips':overall, 'coco_lpips':coco, 'fs2k_lpips':fs2k}
        history.append(entry); progress['last_loss'] = overall
        checkpoint(folder/f"snapshot_{progress['step']:07d}.pt", overall)
        # If the old best.pt was not uploaded, the first new validation becomes
        # the available best. If it improves the old numerical best, replace it.
        if not (folder/'best.pt').is_file() or overall < progress.get('best_loss', float('inf')):
            progress.update(best_loss=overall, best_step=progress['step'])
            checkpoint(folder/'best.pt', overall)
        atomic_json(folder/f"validation_rows_step_{progress['step']:07d}.json", rows)
        atomic_json(folder/'selection.json', dict(progress))
        print('Validation', entry)

    order = list(range(target_updates))
    bar = tqdm(total=target_updates, initial=progress['step'], desc='Resume photo_to_pencil')
    while progress['step'] < target_updates:
        resolution = 256 if progress['step'] < GENERAL_CFG['progressive_512_start'] else 512
        boundary = min(target_updates, GENERAL_CFG['progressive_512_start'] if resolution==256 else target_updates)
        remaining_indices = order[progress['step']:boundary]
        dataset = GeneralBidirectionalDataset(GENERAL_TRAIN_RECORDS, direction, resolution, training=True)
        loader = DataLoader(Subset(dataset,remaining_indices), batch_size=1, shuffle=False,
                            num_workers=2, pin_memory=True, persistent_workers=True)
        generator.train(); discriminator_full.train(); discriminator_half.train()
        for batch in loader:
            source = batch['source'].to(DEVICE, non_blocking=True)
            target = batch['target'].to(DEVICE, non_blocking=True)
            fake = generator(source)
            opt_d.zero_grad(set_to_none=True)
            full_real = discriminator_full(torch.cat([source,target],1))
            full_fake = discriminator_full(torch.cat([source,fake.detach()],1))
            half_source = F.interpolate(source,scale_factor=.5,mode='bilinear',align_corners=False)
            half_target = F.interpolate(target,scale_factor=.5,mode='bilinear',align_corners=False)
            half_fake = F.interpolate(fake.detach(),scale_factor=.5,mode='bilinear',align_corners=False)
            half_real_score = discriminator_half(torch.cat([half_source,half_target],1))
            half_fake_score = discriminator_half(torch.cat([half_source,half_fake],1))
            loss_d = .5*(F.relu(1-full_real).mean()+F.relu(1+full_fake).mean()
                +F.relu(1-half_real_score).mean()+F.relu(1+half_fake_score).mean())
            if not torch.isfinite(loss_d): raise FloatingPointError('Discriminator loss is not finite')
            loss_d.backward(); nn.utils.clip_grad_norm_(discriminator_parameters,1.); opt_d.step(); sched_d.step()
            for discriminator in (discriminator_full,discriminator_half): discriminator.eval().requires_grad_(False)
            opt_g.zero_grad(set_to_none=True)
            paired = paired_direction_loss(fake,target,source,direction)
            full_adversarial = -discriminator_full(torch.cat([source,fake],1)).mean()
            half_generated = F.interpolate(fake,scale_factor=.5,mode='bilinear',align_corners=False)
            half_adversarial = -discriminator_half(torch.cat([half_source,half_generated],1)).mean()
            adversarial = .5*(full_adversarial+half_adversarial)
            next_step = progress['step']+1
            adversarial_weight = GENERAL_CFG['adversarial_max_by_direction'][direction]*min(
                1., next_step/GENERAL_CFG['adversarial_warmup'])
            loss_g = paired+adversarial_weight*adversarial
            if not torch.isfinite(loss_g): raise FloatingPointError('Generator loss is not finite')
            loss_g.backward(); nn.utils.clip_grad_norm_(generator.parameters(),1.,error_if_nonfinite=True)
            opt_g.step(); sched_g.step()
            for discriminator in (discriminator_full,discriminator_half): discriminator.train().requires_grad_(True)
            progress['step'] = next_step
            history.append({'step':next_step,'resolution':resolution,'generator_loss':float(loss_g.detach()),
                'paired_loss':float(paired.detach()),'adversarial_loss':float(adversarial.detach()),
                'adversarial_weight':adversarial_weight,'discriminator_loss':float(loss_d.detach()),
                'lr':sched_g.get_last_lr()[0]})
            bar.update(1); bar.set_postfix(res=resolution,g=round(float(loss_g.detach()),3),d=round(float(loss_d.detach()),3))
            if next_step % GENERAL_CFG['validate_every']==0 or next_step==target_updates: validate()
            if next_step % GENERAL_CFG['save_every']==0 or next_step==target_updates: save_last()
            if next_step==boundary: break
        del loader,dataset
    bar.close(); save_last(); checkpoint(folder/'final.pt',progress['last_loss'])
    atomic_json(folder/'compute.json', {'training_and_validation_seconds':progress['seconds'],
        'hardware':atomic_environment,'best_step':progress['best_step'],'completed_updates':progress['step'],
        'unique_coco_train_images':len(GENERAL_COCO_SPLITS['train']),
        'unique_fs2k_train_pairs':len(SPLITS['train'])})
    del generator,discriminator_full,discriminator_half,opt_g,opt_d,sched_g,sched_d
    gc.collect(); torch.cuda.empty_cache(); return dict(progress)


In [ ]:
# This resumes only the unfinished photo_to_pencil direction.
# Expected remaining time from step 13,500 on a T4: about 70-90 minutes.
PHOTO_PROGRESS = train_resumed_photo_to_pencil()
print('PHOTO TRAINING COMPLETE:', json.dumps(PHOTO_PROGRESS, indent=2))


## 8. Download recovery package

Run this immediately after completion. If you need to stop early, press Stop once, wait for the cell to stop, then run this cell; it saves the most recent 500-step checkpoint.

In [ ]:
# You may run this after completion, or after stopping the training cell safely.
# It downloads a compact recovery ZIP without the large validation snapshots.
from google.colab import files

backup_root = Path('/content/photo_to_pencil_recovery_package')
if backup_root.exists(): shutil.rmtree(backup_root)
backup_root.mkdir(parents=True)
for name in ('last_training_state.pt','best.pt','final.pt','history.json','selection.json','compute.json'):
    source = PHOTO_FOLDER / name
    if source.is_file(): shutil.copy2(source, backup_root/name)
assert (backup_root/'last_training_state.pt').is_file()
archive = shutil.make_archive('/content/photo_to_pencil_recovery_package','zip',root_dir=backup_root)
print('Downloading:', archive)
files.download(archive)


## 9. Visual gate for recovered photo → pencil

In [ ]:
# Optional Section 9-style visual check for the completed photo direction.
best_path = PHOTO_FOLDER/'best.pt'
final_path = PHOTO_FOLDER/'final.pt'
assert best_path.is_file() and final_path.is_file(), 'Complete training first.'
best_model,best_state = load_unet_checkpoint(best_path)
final_model,final_state = load_unet_checkpoint(final_path)
records = GENERAL_COCO_SPLITS['val'][:4] + SPLITS['val'][:4]
fig,axes = plt.subplots(len(records),4,figsize=(15,4*len(records)),squeeze=False)
for row_index,record in enumerate(records):
    photo,sketch = get_general_pair(record,CFG['resolution'])
    images = [photo,sketch,predict_unet(best_model,photo,'photo_to_pencil'),
              predict_unet(final_model,photo,'photo_to_pencil')]
    labels = ['Input photo','Target sketch','Best step '+str(best_state['step']),
              'Final step '+str(final_state['step'])]
    for axis,image,label in zip(axes[row_index],images,labels):
        axis.imshow(image); axis.set_title(label); axis.axis('off')
fig.suptitle('Recovered photo_to_pencil visual gate: COCO scenes then FS2K faces')
fig.tight_layout()
gate_path = SAVE_ROOT/'photo_to_pencil_recovered_visual_gate.png'
fig.savefig(gate_path,dpi=140,bbox_inches='tight'); plt.show(); plt.close(fig)
del best_model,final_model; gc.collect(); torch.cuda.empty_cache()
print('Visual gate saved:',gate_path)
